[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badaouihakimou/machine-learning-notebooks/blob/main/12_numpy_maths_statistiques.ipynb)



# NumPy : mathématiques et statistiques

On sait créer des tableaux et en atteindre des portions. Reste à calculer
dessus sommes, moyennes, produits matriciels sans jamais écrire de boucle.

Le fil conducteur de ce notebook est l'argument `axis`. Il décide si un
calcul se fait sur l'ensemble, par ligne, ou par colonne. C'est le même `axis`
qu'on retrouve dans `concatenate`, dans Pandas, et partout ensuite.

## Le plan

| Section | Le sujet |
|---|---|
| 1 | `axis` : le concept central |
| 2 | Les agrégations : somme, produit, cumul |
| 3 | `argmin` et `argmax` : où plutôt que combien |
| 4 | Trier, et le piège du `None` |
| 5 | Les fonctions mathématiques |
| 6 | Les statistiques, et la variance à deux visages |
| 7 | Les valeurs manquantes |
| 8 | L'algèbre linéaire |

## Trois pièges annoncés

`A.exp()` n'existe pas alors que `A.sum()` existe. La raison n'est pas
arbitraire.

`np.std` divise par n, `statistics.stdev` par n-1 deux résultats différents
pour les mêmes données.

Et un seul `nan` dans un tableau suffit à rendre `nan` toute moyenne calculée
dessus.

Prérequis : les notebooks 10 et 11.

## 1. axis : le concept central

Commençons par un petit tableau qu'on gardera en tête.

In [20]:
import numpy as np

np.random.seed(0)
A = np.random.randint(0, 10, (2, 3))

print(A)
print('forme :', A.shape)

[[5 0 3]
 [3 7 9]]
forme : (2, 3)


In [21]:
print('A.sum() =', A.sum())
print('A.sum(axis=0) =', A.sum(axis=0))
print('A.sum(axis=1) =', A.sum(axis=1))

A.sum() = 27
A.sum(axis=0) = [ 8  7 12]
A.sum(axis=1) = [ 8 19]


Sans `axis`, tout est aggloméré en une seule valeur.

Avec `axis=0`, on obtient trois valeurs une par colonne. Avec `axis=1`, deux
valeurs une par ligne.

### Comment ne plus se tromper

La formulation qui marche : `axis=i` est l'axe qui disparaît.

Le tableau a la forme `(2, 3)`. Sommer sur `axis=0` supprime la première
dimension, il reste `(3,)`. Sommer sur `axis=1` supprime la seconde, il reste
`(2,)`.

In [22]:
print('A :', A.shape)
print('axis=0 -> :', A.sum(axis=0).shape, ' l\'axe 0 a disparu')
print('axis=1 -> :', A.sum(axis=1).shape, ' l\'axe 1 a disparu')

A : (2, 3)
axis=0 -> : (3,)  l'axe 0 a disparu
axis=1 -> : (2,)  l'axe 1 a disparu


Autre façon de le dire : `axis=0` parcourt vers le bas, donc agrège les
lignes entre elles et produit un résultat par colonne.

C'est contre-intuitif au début, parce qu'on dit « axis=0 » et qu'on obtient des
colonnes. Mais c'est cohérent avec `concatenate` du notebook 10 : `axis=0` y
faisait grandir le nombre de lignes.

| | Ce qu'on agrège | Ce qu'on obtient |
|---|---|---|
| `axis=0` | les lignes entre elles | une valeur par colonne |
| `axis=1` | les colonnes entre elles | une valeur par ligne |

### Garder la dimension

`keepdims=True` évite que l'axe disparaisse. Utile pour la suite des calculs.

In [23]:
print('sans keepdims :', A.sum(axis=1).shape)
print('avec keepdims :', A.sum(axis=1, keepdims=True).shape)
print()
print(A.sum(axis=1, keepdims=True))

sans keepdims : (2,)
avec keepdims : (2, 1)

[[ 8]
 [19]]


La forme `(2, 1)` au lieu de `(2,)` permet de diviser directement chaque ligne
par sa somme, grâce au broadcasting du notebook 13 :

```python
A / A.sum(axis=1, keepdims=True) # chaque ligne somme à 1
```

Sans `keepdims`, ce calcul échoue ou donne un résultat faux. C'est le piège
`(2,)` contre `(2, 1)` qui revient.

## 2. Les agrégations

In [24]:
print('somme :', A.sum())
print('produit :', A.prod())
print('minimum :', A.min())
print('maximum :', A.max())
print('moyenne :', A.mean().round(3))

somme : 27
produit : 0
minimum : 0
maximum : 9
moyenne : 4.5


Toutes acceptent `axis` :

In [25]:
print('min par colonne :', A.min(axis=0))
print('min par ligne :', A.min(axis=1))
print('max par colonne :', A.max(axis=0))

min par colonne : [3 0 3]
min par ligne : [0 3]
max par colonne : [5 7 9]


### Les cumuls

`cumsum` et `cumprod` ne réduisent pas : ils renvoient un tableau de même taille,
contenant les totaux partiels.

In [26]:
print('A :'); print(A)
print()
print('cumsum() aplati :', A.cumsum())
print()
print('cumsum(axis=0), cumul vers le bas :')
print(A.cumsum(axis=0))
print()
print('cumsum(axis=1), cumul vers la droite :')
print(A.cumsum(axis=1))

A :
[[5 0 3]
 [3 7 9]]

cumsum() aplati : [ 5  5  8 11 18 27]

cumsum(axis=0), cumul vers le bas :
[[ 5  0  3]
 [ 8  7 12]]

cumsum(axis=1), cumul vers la droite :
[[ 5  5  8]
 [ 3 10 19]]


Sans `axis`, le tableau est d'abord aplati c'est pourquoi le résultat est à une
dimension.

`cumsum` sert dès qu'on veut une évolution : un solde bancaire au fil des
opérations, une distribution cumulée, une somme glissante.

Attention à `cumprod` sur de grands tableaux : les produits explosent vite et
dépassent la capacité du type, comme vu au notebook 10.

In [27]:
grand = np.arange(1, 25)
print('cumprod en int64 :', grand.cumprod()[-3:])
print('vraie valeur de 24! :', 620448401733239439360000)

cumprod en int64 : [-1250660718674968576  8128291617894825984 -7835185981329244160]
vraie valeur de 24! : 620448401733239439360000


Les dernières valeurs sont fausses : dépassement silencieux de `int64`. Aucune
erreur, juste des nombres absurdes.

## 3. argmin et argmax

`min` donne la valeur la plus petite. `argmin` donne sa position.

In [28]:
print(A)
print()
print('min :', A.min(), ' argmin :', A.argmin())
print('max :', A.max(), ' argmax :', A.argmax())

[[5 0 3]
 [3 7 9]]

min : 0  argmin : 1
max : 9  argmax : 5


Sans `axis`, la position est calculée sur le tableau aplati. Ici `argmin`
renvoie 1, ce qui correspond à la case `[0, 1]`.

Pour retrouver les coordonnées à deux dimensions :

In [29]:
position = np.unravel_index(A.argmin(), A.shape)
print('position 2D :', position)
print('vérification :', A[position])

position 2D : (np.int64(0), np.int64(1))
vérification : 0


Avec `axis`, on obtient une position par ligne ou par colonne :

In [30]:
print('argmin(axis=0) :', A.argmin(axis=0), ' -> l\'indice de ligne du min de chaque colonne')
print('argmin(axis=1) :', A.argmin(axis=1), ' -> l\'indice de colonne du min de chaque ligne')

argmin(axis=0) : [1 0 0]  -> l'indice de ligne du min de chaque colonne
argmin(axis=1) : [1 0]  -> l'indice de colonne du min de chaque ligne


### Pourquoi c'est essentiel en machine learning

Un modèle de classification renvoie une probabilité par classe. La prédiction est
la classe la plus probable donc `argmax`.

In [31]:
probabilites = np.array([[0.1, 0.7, 0.2],
                         [0.8, 0.1, 0.1],
                         [0.2, 0.3, 0.5]])

print('Probabilités par classe :')
print(probabilites)
print()
print('Classe prédite :', probabilites.argmax(axis=1))
print('Confiance :', probabilites.max(axis=1))

Probabilités par classe :
[[0.1 0.7 0.2]
 [0.8 0.1 0.1]
 [0.2 0.3 0.5]]

Classe prédite : [1 0 2]
Confiance : [0.7 0.8 0.5]


`axis=1` parce qu'on cherche le maximum sur chaque ligne, c'est-à-dire pour
chaque individu.

C'est exactement ce que fait `model.predict()` en scikit-learn à partir de
`model.predict_proba()`.

## 4. Trier

### Le piège du None

In [32]:
resultat = np.array([1, 3, -2, 5]).sort()
print('np.array([...]).sort() renvoie :', resultat)

np.array([...]).sort() renvoie : None


`sort()` est une méthode : elle trie sur place et renvoie `None`. C'est la
convention Python du notebook 04, celle de `liste.sort()`.

Le problème, dans l'écriture ci-dessus, est qu'on trie un tableau temporaire
qu'aucune variable ne retient. Le tri a bien lieu, puis tout est perdu.

Deux façons correctes :

In [33]:
tableau = np.array([1, 3, -2, 5])
tableau.sort() # trie sur place
print('méthode sort   :', tableau)

tableau = np.array([1, 3, -2, 5])
trie = np.sort(tableau) # renvoie une copie triée
print('fonction sort  :', trie, '  original :', tableau)

méthode sort   : [-2  1  3  5]
fonction sort  : [-2  1  3  5]   original : [ 1  3 -2  5]


| | Effet | Renvoie |
|---|---|---|
| `tableau.sort()` | trie sur place | `None` |
| `np.sort(tableau)` | ne modifie pas | une copie triée |

Même distinction que `sort` et `sorted` en Python pur.

### argsort : l'ordre des indices

In [34]:
valeurs = np.array([1, 3, -2, 5])
ordre = valeurs.argsort()

print('valeurs :', valeurs)
print('argsort :', ordre)
print()
print('Lecture : le plus petit est en position', ordre[0],
      ', puis en position', ordre[1])
print('Reconstruction :', valeurs[ordre])

valeurs : [ 1  3 -2  5]
argsort : [2 0 1 3]

Lecture : le plus petit est en position 2 , puis en position 0
Reconstruction : [-2  1  3  5]


`argsort` renvoie les indices qui trieraient le tableau. Passer ces indices
entre crochets donne le tableau trié c'est du fancy indexing du notebook 11.

L'intérêt est de trier un tableau selon un autre :

In [35]:
noms = np.array(['Ali', 'Sara', 'Yanis', 'Nour'])
notes = np.array([12, 18, 9, 15])

classement = notes.argsort()[::-1] # décroissant

for rang, i in enumerate(classement, start=1):
    print(f'{rang}. {noms[i]:<8} {notes[i]}')

1. Sara     18
2. Nour     15
3. Ali      12
4. Yanis    9


C'est le mécanisme derrière `sort_values` en Pandas, et derrière tout classement
par score en machine learning.

## 5. Les fonctions mathématiques

### Le piège : méthode ou fonction ?

In [36]:
try:
    A.exp()
except AttributeError as e:
    print('AttributeError :', e)

print()
print('np.exp(A) fonctionne :')
print(np.exp(A).round(2))

AttributeError : 'numpy.ndarray' object has no attribute 'exp'

np.exp(A) fonctionne :
[[1.48410e+02 1.00000e+00 2.00900e+01]
 [2.00900e+01 1.09663e+03 8.10308e+03]]


`A.sum()` existe mais `A.exp()` non. Ce n'est pas arbitraire.

Les agrégations somme, moyenne, minimum sont des méthodes de `ndarray`,
parce qu'elles décrivent le tableau lui-même.

Les fonctions élément par élément exponentielle, logarithme, sinus sont
des fonctions de NumPy. Elles s'appliquent à n'importe quoi, y compris à une
liste Python :

```python
np.exp([1, 2, 3]) # fonctionne sur une liste
```

En cas de doute, `np.fonction(tableau)` marche presque toujours ; l'écriture
`tableau.fonction()` est réservée aux agrégations.

In [37]:
petit = np.array([[1, 2], [3, 4]])

print('exp :'); print(np.exp(petit).round(2))
print()
print('sqrt :', np.sqrt(petit).ravel().round(3))
print('sin :', np.sin(petit).ravel().round(3))
print('abs :', np.abs(np.array([-1, 2, -3])))
print('round :', np.round(np.array([1.234, 5.678]), 1))

exp :
[[ 2.72  7.39]
 [20.09 54.6 ]]

sqrt : [1.    1.414 1.732 2.   ]
sin : [ 0.841  0.909  0.141 -0.757]
abs : [1 2 3]
round : [1.2 5.7]


Ces fonctions sont vectorisées : elles s'appliquent à chaque élément sans
boucle. C'est le principe du notebook 10.

### Le piège du logarithme

`np.log` de zéro ou d'un négatif ne lève pas d'erreur il produit `-inf` ou
`nan`, avec un simple avertissement.

In [38]:
with np.errstate(all='ignore'):
    resultat = np.log(np.array([0, 1, -2, 10]))

print(resultat)

[      -inf 0.                nan 2.30258509]


C'est un problème réel : le tableau `A` du début contient un zéro, donc
`np.log(A)` produit un `-inf` qui contaminera tous les calculs suivants.

La parade habituelle consiste à ajouter une petite constante :

```python
np.log(A + 1e-10)
```

C'est exactement ce qu'on fait en machine learning pour la fonction de coût
logarithmique, où une probabilité de zéro donnerait un coût infini.

Pour être averti au lieu de subir, on peut demander une vraie erreur :

```python
np.seterr(divide='raise', invalid='raise')
```

## 6. Les statistiques

In [41]:
np.random.seed(0)
A = np.random.randint(0, 10, (5, 5))

print(A)
print()
print('moyenne :', A.mean().round(3))
print('médiane :', np.median(A))
print('écart-type:', A.std().round(3))
print('variance :', A.var().round(3))

[[5 0 3 3 7]
 [9 3 5 2 4]
 [7 6 8 8 1]
 [6 7 7 8 1]
 [5 9 8 9 4]]

moyenne : 5.4
médiane : 6.0
écart-type: 2.668
variance : 7.12


Note que `median` est une fonction et non une méthode, contrairement à
`mean`. C'est une irrégularité de NumPy, sans logique particulière.

### Le piège de la variance

`np.var` divise par n. `statistics.variance`, vu au notebook 08, divise par n-1.

In [42]:
donnees = np.array([2, 4, 4, 4, 5, 5, 7, 9])

print('np.var (population, n) :', donnees.var())
print('np.var (échantillon, n-1) :', donnees.var(ddof=1))
print()
print('np.std (n) :', donnees.std().round(4))
print('np.std (n-1) :', donnees.std(ddof=1).round(4))

np.var (population, n) : 4.0
np.var (échantillon, n-1) : 4.571428571428571

np.std (n) : 2.0
np.std (n-1) : 2.1381


Deux valeurs différentes, aucune n'est fausse.

`ddof` signifie *delta degrees of freedom* : le diviseur est `n - ddof`.

| Contexte | Convention |
|---|---|
| NumPy par défaut | `ddof=0`, division par n |
| `statistics` et Pandas | division par n-1 |
| scikit-learn | `ddof=0` |

La correction n-1 s'applique quand les données sont un échantillon d'une
population plus large dont on veut estimer la dispersion.

Sur huit valeurs, l'écart est de 7 %. Sur mille, il est négligeable. Mais si tu
compares un résultat NumPy à un résultat Pandas, tu chercheras longtemps.

### Les corrélations

In [43]:
np.random.seed(0)
X = np.random.randn(100, 3)
X[:, 1] = X[:, 0] * 2 + np.random.randn(100) * 0.1 # colonne 1 liée à la 0

print('Corrélations entre COLONNES :')
print(np.corrcoef(X.T).round(3))

Corrélations entre COLONNES :
[[1.    0.999 0.111]
 [0.999 1.    0.113]
 [0.111 0.113 1.   ]]


Le `.T` est indispensable, et c'est le même piège qu'au notebook Pandas.

`np.corrcoef` calcule les corrélations entre les lignes du tableau qu'on lui
donne. Or dans `X`, les lignes sont les individus et les colonnes les variables.

Sans transposition, on obtiendrait une matrice 100×100 de corrélations entre
individus, ce qui n'a aucun sens.

In [44]:
print('sans .T :', np.corrcoef(X).shape, '-> corrélations entre individus')
print('avec .T :', np.corrcoef(X.T).shape, '-> corrélations entre variables')

sans .T : (100, 100) -> corrélations entre individus
avec .T : (3, 3) -> corrélations entre variables


Une corrélation vaut entre -1 et 1. La diagonale vaut toujours 1 : chaque
variable est parfaitement corrélée avec elle-même.

Ici, les colonnes 0 et 1 ont une corrélation proche de 1 puisqu'on les a
construites liées.

### Compter les valeurs

In [45]:
np.random.seed(0)
A = np.random.randint(0, 10, (5, 5))

valeurs, effectifs = np.unique(A, return_counts=True)

print('valeurs :', valeurs)
print('effectifs :', effectifs)

valeurs : [0 1 2 3 4 5 6 7 8 9]
effectifs : [1 2 1 3 2 3 2 4 4 3]


`np.unique` renvoie les valeurs distinctes, triées. Avec `return_counts=True`, il
renvoie un tuple de deux tableaux, qu'on déballe.

Pour classer du plus fréquent au moins fréquent, on combine avec `argsort` :

In [46]:
ordre = effectifs.argsort()[::-1]

for valeur, effectif in zip(valeurs[ordre], effectifs[ordre]):
    print(f'la valeur {valeur} apparaît {effectif} fois')

la valeur 8 apparaît 4 fois
la valeur 7 apparaît 4 fois
la valeur 9 apparaît 3 fois
la valeur 5 apparaît 3 fois
la valeur 3 apparaît 3 fois
la valeur 6 apparaît 2 fois
la valeur 4 apparaît 2 fois
la valeur 1 apparaît 2 fois
la valeur 2 apparaît 1 fois
la valeur 0 apparaît 1 fois


Le `[::-1]` inverse pour obtenir l'ordre décroissant `argsort` trie toujours
en croissant.

C'est l'équivalent NumPy du `value_counts()` de Pandas et du `Counter` du
notebook 07.

## 7. Les valeurs manquantes

`nan` signifie *not a number*. C'est une valeur flottante spéciale qui représente
une donnée absente ou un calcul impossible.

In [47]:
np.random.seed(0)
B = np.random.randn(5, 5)
B[0, 2] = np.nan
B[4, 3] = np.nan

print(B.round(2))

[[ 1.76  0.4    nan  2.24  1.87]
 [-0.98  0.95 -0.15 -0.1   0.41]
 [ 0.14  1.45  0.76  0.12  0.44]
 [ 0.33  1.49 -0.21  0.31 -0.85]
 [-2.55  0.65  0.86   nan  2.27]]


### Un seul nan contamine tout

In [49]:
print('B.mean() :', B.mean())
print('B.sum() :', B.sum())
print('B.max() :', B.max())

B.mean() : nan
B.sum() : nan
B.max() : nan


Tout devient `nan`. C'est logique la moyenne d'un ensemble contenant une valeur
inconnue est inconnue mais c'est déroutant quand on ne s'y attend pas.

Les versions qui ignorent les `nan` existent, préfixées par `nan` :

In [50]:
print('nanmean :', np.nanmean(B).round(4))
print('nansum :', np.nansum(B).round(4))
print('nanmax :', np.nanmax(B).round(4))
print('nanstd :', np.nanstd(B).round(4))
print('nanmedian :', np.nanmedian(B).round(4))

nanmean : 0.5062
nansum : 11.6428
nanmax : 2.2698
nanstd : 1.0821
nanmedian : 0.4106


### Détecter les nan

Attention, un `nan` n'est égal à rien, pas même à lui-même.

In [51]:
print('np.nan == np.nan :', np.nan == np.nan)
print('B[0, 2] == np.nan :', B[0, 2] == np.nan)
print()
print('np.isnan(B[0, 2]) :', np.isnan(B[0, 2]))

np.nan == np.nan : False
B[0, 2] == np.nan : False

np.isnan(B[0, 2]) : True


C'est défini par la norme IEEE 754 : une valeur inconnue ne peut pas être déclarée
égale à une autre valeur inconnue.

La conséquence pratique : `== np.nan` ne fonctionne jamais. Il faut
`np.isnan`.

In [53]:
print('Masque des nan :')
print(np.isnan(B))
print()
print('Combien :', np.isnan(B).sum())
print('Proportion :', (np.isnan(B).sum() / B.size).round(3))
print('Par colonne :', np.isnan(B).sum(axis=0))

Masque des nan :
[[False False  True False False]
 [False False False False False]
 [False False False False False]
 [False False False False False]
 [False False False  True False]]

Combien : 2
Proportion : 0.08
Par colonne : [0 0 1 1 0]


`isnan` renvoie un masque booléen, sur lequel `sum` compte les `True` l'idiome
du notebook 11.

Le compte par colonne est le premier réflexe sur un vrai jeu de données : c'est
l'équivalent du `data.isna().sum()` de Pandas.

### Remplacer

In [54]:
propre = B.copy() # copy : on garde l'original
propre[np.isnan(propre)] = 0

print('nan restants :', np.isnan(propre).sum())

nan restants : 0


Remplacer par zéro est le choix le plus simple, et rarement le meilleur : cela
déplace la moyenne vers zéro.

La médiane est généralement préférable, comme démontré au notebook Pandas :

In [55]:
propre = B.copy()

for j in range(propre.shape[1]):
    colonne = propre[:, j] # vue
    colonne[np.isnan(colonne)] = np.nanmedian(colonne)

print('nan restants :', np.isnan(propre).sum())
print('moyenne avant (nanmean) :', np.nanmean(B).round(4))
print('moyenne après :', propre.mean().round(4))

nan restants : 0
moyenne avant (nanmean) : 0.5062
moyenne après : 0.4866


Et le rappel du notebook Pandas : avant de remplacer ou de supprimer, il faut se
demander pourquoi la valeur manque. Sur le Titanic, les âges manquants
n'étaient pas répartis au hasard, et les supprimer faussait le taux de survie.

## 8. L'algèbre linéaire

### La transposée

In [56]:
A = np.ones((2, 3))

print('A :', A.shape)
print('A.T :', A.T.shape)

A : (2, 3)
A.T : (3, 2)


`.T` échange lignes et colonnes. C'est un attribut, pas une méthode : pas de
parenthèses.

Et c'est une vue elle ne copie rien, elle change juste la façon de lire la
mémoire.

In [57]:
A = np.array([[1, 2, 3], [4, 5, 6]])
print('T est une vue :', A.T.base is A)

T est une vue : True


### Le produit matriciel

À ne pas confondre avec la multiplication élément par élément.

In [58]:
A = np.ones((2, 3))
B = np.ones((3, 2))

print('A.dot(B) :', A.dot(B).shape)
print(A.dot(B))
print()
print('B.dot(A) :', B.dot(A).shape)

A.dot(B) : (2, 2)
[[3. 3.]
 [3. 3.]]

B.dot(A) : (3, 3)


`(2,3) · (3,2)` donne `(2,2)`. `(3,2) · (2,3)` donne `(3,3)`. Le produit
matriciel n'est pas commutatif l'ordre change le résultat, et même sa forme.

La règle des dimensions : le nombre de colonnes du premier doit égaler le nombre
de lignes du second.

In [59]:
try:
    np.ones((2, 3)).dot(np.ones((2, 3)))
except ValueError as e:
    print('ValueError :', str(e)[:80])

ValueError : shapes (2,3) and (2,3) not aligned: 3 (dim 1) != 2 (dim 0)


Ce message est l'un des plus fréquents en machine learning. La lecture est
toujours la même : afficher les deux formes et vérifier que les dimensions
intérieures coïncident.

### Trois écritures équivalentes

In [61]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print('A.dot(B) :'); print(A.dot(B))
print()
print('A @ B :'); print(A @ B)
print()
print('A * B  -> élément par élément, PAS un produit matriciel :')
print(A * B)

A.dot(B) :
[[19 22]
 [43 50]]

A @ B :
[[19 22]
 [43 50]]

A * B  -> élément par élément, PAS un produit matriciel :
[[ 5 12]
 [21 32]]


L'opérateur `@` est la façon moderne d'écrire le produit matriciel, plus lisible
que `.dot`.

`*` fait tout autre chose : il multiplie case par case. Confondre les deux est
une erreur classique, et elle ne lève aucune erreur quand les formes
coïncident juste un résultat faux.

### Déterminant, inverse, valeurs propres

In [62]:
np.random.seed(0)
M = np.random.randint(0, 10, (3, 3))

print(M)
print()
print('déterminant :', round(np.linalg.det(M), 4))
print()
print('inverse :')
print(np.linalg.inv(M).round(3))
print()
print('vérification M @ M⁻¹ = I :', np.allclose(M @ np.linalg.inv(M), np.eye(3)))

[[5 0 3]
 [3 7 9]
 [3 5 2]]

déterminant : -173.0

inverse :
[[ 0.179 -0.087  0.121]
 [-0.121 -0.006  0.208]
 [ 0.035  0.145 -0.202]]

vérification M @ M⁻¹ = I : True


Le `np.allclose` plutôt qu'un `==` : le produit ne donne pas exactement la matrice
identité à cause des arrondis flottants. C'est le piège du notebook 02, et
`allclose` est son remède vectorisé.

### Quand l'inverse n'existe pas

In [63]:
singuliere = np.array([[1, 2],
                       [2, 4]]) # la 2e ligne est le double de la 1re

print('déterminant :', np.linalg.det(singuliere))

try:
    np.linalg.inv(singuliere)
except np.linalg.LinAlgError as e:
    print('LinAlgError :', e)

déterminant : 0.0
LinAlgError : Singular matrix


Un déterminant nul signifie que les lignes sont liées entre elles l'une
n'apporte aucune information nouvelle. La matrice n'est alors pas inversible.

C'est exactement ce qui arrive en régression linéaire quand deux variables sont
parfaitement corrélées. D'où l'intérêt de la matrice de corrélation vue en
section 6 : elle permet de repérer le problème avant qu'il ne survienne.

`pinv`, la pseudo-inverse, fonctionne même dans ce cas :

In [64]:
print('pseudo-inverse :')
print(np.linalg.pinv(singuliere).round(4))

pseudo-inverse :
[[0.04 0.08]
 [0.08 0.16]]


Elle donne la meilleure approximation au sens des moindres carrés. C'est ce que
scikit-learn utilise en interne pour la régression linéaire, précisément pour ne
pas échouer sur des données corrélées.

### Valeurs et vecteurs propres

In [65]:
valeurs_propres, vecteurs_propres = np.linalg.eig(M)

print('valeurs propres :', valeurs_propres.round(3))
print()
print('vecteurs propres (en colonnes) :')
print(vecteurs_propres.round(3))

valeurs propres : [12.491  4.552 -3.043]

vecteurs propres (en colonnes) :
[[-0.186 -0.822 -0.282]
 [-0.865  0.556 -0.592]
 [-0.466  0.123  0.755]]


`eig` renvoie un tuple de deux éléments, qu'il faut déballer. L'afficher sans
déballage donne quelque chose d'illisible.

Les valeurs propres peuvent être complexes, même pour une matrice réelle c'est
normal, et c'est pourquoi NumPy affiche parfois des `+0.j`.

C'est le fondement mathématique de l'analyse en composantes principales, qui
cherche les directions de plus grande variance dans un nuage de points. Le
notebook 24 y reviendra.

## 9. Mémo

### axis

| | Ce qu'on agrège | Résultat sur `(2, 3)` |
|---|---|---|
| pas d'axis | tout | une valeur |
| `axis=0` | les lignes | `(3,)`, une par colonne |
| `axis=1` | les colonnes | `(2,)`, une par ligne |

`axis=i` est l'axe qui disparaît. `keepdims=True` le conserve en taille 1.

### Méthode ou fonction

| Méthode de `ndarray` | Fonction NumPy |
|---|---|
| `A.sum()`, `A.mean()` | `np.exp(A)`, `np.log(A)` |
| `A.min()`, `A.max()` | `np.sqrt(A)`, `np.sin(A)` |
| `A.std()`, `A.var()` | `np.median(A)`, `np.unique(A)` |
| `A.argmin()`, `A.sort()` | `np.sort(A)`, `np.corrcoef(A)` |

Les agrégations sont des méthodes, les opérations élément par élément des
fonctions. `np.median` est une exception.

### Les valeurs manquantes

| Besoin | Fonction |
|---|---|
| Détecter | `np.isnan(A)` |
| Compter | `np.isnan(A).sum()` |
| Calculer malgré tout | `np.nanmean`, `np.nansum`, `np.nanstd` |

`A == np.nan` ne fonctionne jamais.

### Algèbre linéaire

| Écriture | Effet |
|---|---|
| `A.T` | transposée, une vue |
| `A @ B` ou `A.dot(B)` | produit matriciel |
| `A * B` | élément par élément |
| `np.linalg.det(A)` | déterminant |
| `np.linalg.inv(A)` | inverse |
| `np.linalg.pinv(A)` | pseudo-inverse |
| `np.linalg.eig(A)` | valeurs et vecteurs propres |

### Les pièges

| Situation | Ce qui se passe |
|---|---|
| `A.exp()` | n'existe pas, il faut `np.exp(A)` |
| `np.array([...]).sort()` | trie un temporaire, renvoie `None` |
| `np.var` sans `ddof` | divise par n, pas n-1 |
| `np.corrcoef(X)` sans `.T` | corrèle les individus, pas les variables |
| Un `nan` dans le tableau | toute agrégation devient `nan` |
| `== np.nan` | toujours `False` |
| `A * B` pour un produit matriciel | résultat faux, sans erreur |
| `cumprod` sur de grands nombres | dépassement silencieux |
| `np.log(0)` | `-inf`, simple avertissement |

## 10. Exercices

**Exercice 1**

Écris une fonction `standardiser(X)` qui centre et réduit chaque colonne d'un
tableau : soustraire la moyenne, diviser par l'écart-type. Vérifie que chaque
colonne obtenue a bien une moyenne de 0 et un écart-type de 1.

Attention à deux choses : le bon `axis`, et le cas d'une colonne constante.

**Exercice 2**

À partir d'un tableau `(1000, 4)` contenant 5 % de valeurs manquantes placées au
hasard, écris une fonction `rapport(X)` qui affiche, colonne par colonne : le
nombre de `nan`, la moyenne, la médiane et l'écart-type calculés sans eux.

Compare ensuite `X.mean(axis=0)` et `np.nanmean(X, axis=0)`, et explique.

**Exercice 3**

Cette fonction contient trois pièges de ce notebook :

```python
def resumer(X):
    correlations = np.corrcoef(X)
    moyennes = X.mean(axis=1)
    ecarts = X.std()
    meilleure = X.argmax()
    return correlations, moyennes, ecarts, meilleure
```

Applique-la à un tableau `(50, 3)` représentant 50 individus et 3 variables.
Explique pourquoi chacune des quatre lignes ne fait pas ce qu'on attend, puis
corrige la fonction.

## Pour continuer

Le notebook suivant porte sur le broadcasting la règle qui permet à NumPy
d'additionner des tableaux de formes différentes. C'est ce qui explique pourquoi
`keepdims=True` était nécessaire en section 1, et pourquoi `(3,)` et `(3, 1)` ne
se comportent pas pareil.

C'est le dernier notebook NumPy, et probablement le plus utile : il élimine la
majorité des erreurs de dimension.

In [66]:
# Exercice 1

In [67]:
def standardiser(X):
    """Centre et réduit chaque colonne : moyenne 0, écart-type 1.
    Une colonne constante est laissée à zéro plutôt que de diviser par zéro.
    Retourne un nouveau tableau ; X n'est pas modifié.
    """
    X = X.astype(float) # copie ET conversion
    moyennes = X.mean(axis=0) # une valeur par colonne
    ecarts = X.std(axis=0)
    ecarts = np.where(ecarts == 0, 1, ecarts) # évite la division par zéro
    return (X - moyennes) / ecarts

In [70]:
rng = np.random.default_rng(0)
X = rng.normal(5, 3, size=(100, 4))
X[:, 3] = 7 # une colonne constante
Z = standardiser(X)
print('moyennes :', Z.mean(axis=0).round(10))
print('écarts :', Z.std(axis=0).round(10))

moyennes : [-0.  0. -0.  0.]
écarts : [1. 1. 1. 0.]


In [71]:
print('axis=0 :', X.mean(axis=0).shape, '-> une valeur par variable')
print('axis=1 :', X.mean(axis=1).shape, '-> une valeur par individu')

axis=0 : (4,) -> une valeur par variable
axis=1 : (100,) -> une valeur par individu


In [72]:
constante = np.full((10, 1), 5.0)
avec_bug = (constante - constante.mean(axis=0)) / constante.std(axis=0)
print(avec_bug.ravel())

[nan nan nan nan nan nan nan nan nan nan]


/tmp/ipykernel_877/271793841.py:2: RuntimeWarning: invalid value encountered in divide
  avec_bug = (constante - constante.mean(axis=0)) / constante.std(axis=0)


In [73]:
from sklearn.preprocessing import StandardScaler
Z_sklearn = StandardScaler().fit_transform(X)
print('identique :', np.allclose(Z, Z_sklearn))

identique : True


In [74]:
# Exercice 2

In [75]:
 def rapport(X, noms=None):
    """Affiche un résumé colonne par colonne, en ignorant les nan."""
    if noms is None:
        noms = [f'col_{j}' for j in range(X.shape[1])]

    print(f'{"variable":<10} {"nan":>5} {"%":>7} {"moyenne":>9} '
          f'{"médiane":>9} {"écart-type":>11}')
    print('-' * 55)

    for j, nom in enumerate(noms):
        colonne = X[:, j]
        n_nan = np.isnan(colonne).sum()

        print(f'{nom:<10} {n_nan:>5} {n_nan/len(colonne):>7.1%} '
              f'{np.nanmean(colonne):>9.3f} {np.nanmedian(colonne):>9.3f} '
              f'{np.nanstd(colonne):>11.3f}')

In [76]:
rng = np.random.default_rng(0)

X = rng.normal(size=(1000, 4))
masque = rng.random(X.shape) < 0.05 # 5 % des cases
X[masque] = np.nan

print('nan au total :', np.isnan(X).sum(), 'sur', X.size)
print()
rapport(X)

nan au total : 186 sur 4000

variable     nan       %   moyenne   médiane  écart-type
-------------------------------------------------------
col_0         47    4.7%     0.007     0.021       0.990
col_1         53    5.3%    -0.029    -0.044       0.975
col_2         41    4.1%    -0.045    -0.053       1.020
col_3         45    4.5%     0.016    -0.040       1.022


In [77]:
print('X.mean(axis=0) :', X.mean(axis=0))
print('np.nanmean(X, axis=0):', np.nanmean(X, axis=0).round(4))

X.mean(axis=0) : [nan nan nan nan]
np.nanmean(X, axis=0): [ 0.0073 -0.0288 -0.0453  0.0156]


In [78]:
for j in range(4):
    presentes = (~np.isnan(X[:, j])).sum()
    print(f'colonne {j} : {presentes} valeurs utilisées')

colonne 0 : 953 valeurs utilisées
colonne 1 : 947 valeurs utilisées
colonne 2 : 959 valeurs utilisées
colonne 3 : 955 valeurs utilisées


In [79]:
# Exercice 3

In [80]:
def resumer(X):
    correlations = np.corrcoef(X)
    moyennes = X.mean(axis=1)
    ecarts = X.std()
    meilleure = X.argmax()
    return correlations, moyennes, ecarts, meilleure

In [81]:
rng = np.random.default_rng(0)
X = rng.normal(size=(50, 3)) # 50 individus, 3 variables

correlations, moyennes, ecarts, meilleure = resumer(X)

print('correlations :', correlations.shape)
print('moyennes :', moyennes.shape)
print('ecarts :', ecarts, type(ecarts))
print('meilleure :', meilleure)

correlations : (50, 50)
moyennes : (50,)
ecarts : 0.9586410181646288 <class 'numpy.float64'>
meilleure : 79


In [82]:
print('argmax aplati :', X.argmax())
print('coordonnées :', np.unravel_index(X.argmax(), X.shape))

argmax aplati : 79
coordonnées : (np.int64(26), np.int64(1))


In [83]:
def resumer(X, ddof=1):
    """Résumé statistique d'un tableau (individus en lignes, variables en colonnes).
    Retourne un dictionnaire pour que chaque valeur soit nommée.
    """
    return {
        'correlations': np.corrcoef(X.T), # .T : entre VARIABLES
        'moyennes': X.mean(axis=0), # axis=0 : par variable
        'ecarts': X.std(axis=0, ddof=ddof), # par variable, n-1
        'position_max': np.unravel_index(X.argmax(), X.shape), # coordonnées 2D
        'valeur_max': X.max(),
    }

In [84]:
resultat = resumer(X)

print('corrélations :', resultat['correlations'].shape)
print(resultat['correlations'].round(3))
print()
print('moyennes :', resultat['moyennes'].round(4))
print('écarts-types :', resultat['ecarts'].round(4))
print('position max :', resultat['position_max'], '-> individu, variable')
print('valeur max :', resultat['valeur_max'].round(4))
print('vérification :', X[resultat['position_max']].round(4))

corrélations : (3, 3)
[[ 1.     0.101  0.182]
 [ 0.101  1.    -0.059]
 [ 0.182 -0.059  1.   ]]

moyennes : [-0.1982  0.0895  0.2978]
écarts-types : [1.1176 0.8382 0.8573]
position max : (np.int64(26), np.int64(1)) -> individu, variable
valeur max : 2.0024
vérification : 2.0024


In [85]:
print(X.shape) # (50, 3) : 50 individus, 3 variables
print(np.corrcoef(X).shape) # (50, 50) : suspect
print(np.corrcoef(X.T).shape) # (3, 3) : cohérent

(50, 3)
(50, 50)
(3, 3)
